# Module 2 Practical — Prompt Engineering Lab 🪄

**PMS RoBoTics Research Center · 4-Week Internship & Training Program**
**Week 1 · Module 2: Prompt engineering — writing effective prompts for real-world tasks**

---

### What you'll do in the next 30 minutes

| # | Experiment | Technique |
|---|-----------|-----------|
| 1 | Vague vs engineered | The 5-part prompt anatomy |
| 2 | One question, three personas | Role prompting |
| 3 | Support-ticket classifier | Few-shot prompting |
| 4 | The discount puzzle | Step-by-step reasoning |
| 5 | Email → JSON extractor | Structured output |
| 🏁 | **SmartBot v2** | Everything combined |

> 💡 **Prompt anatomy checklist** (from today's theory): **Role · Task · Context · Format · Constraints**. Every failing prompt is missing one of these five.

## Step 0 — Setup (same as yesterday, 2 minutes)

Use the same free Gemini API key from Module 1. If you lost it, create a new one at [aistudio.google.com](https://aistudio.google.com) → "Get API key".

In [ ]:
%pip install -q -U google-genai

In [ ]:
from getpass import getpass
from google import genai
from google.genai import types

API_KEY = getpass("Paste your Gemini API key and press Enter: ")
client = genai.Client(api_key=API_KEY)
MODEL = "gemini-2.0-flash"

# small helper so our experiments stay tidy
def ask(prompt, system=None):
    config = types.GenerateContentConfig(system_instruction=system) if system else None
    response = client.models.generate_content(model=MODEL, contents=prompt, config=config)
    return response.text.strip()

print("✅ Ready:", ask("Reply with exactly: prompt lab online"))

---
## Experiment 1 — Vague vs engineered prompt ⚖️

Same model, same product. The only variable is the prompt. Watch what each of the five anatomy parts buys you:

In [ ]:
vague = "Write about our kettle."

engineered = """You are a copywriter for SmartMart, a friendly retail store in Pune.   # ROLE
Write a product description of the Prima 1.5L electric kettle for our website.        # TASK
Facts you may use (do not invent others):                                              # CONTEXT
- 1.5 litre capacity, 1500W fast boil
- auto shut-off when boiling
- 1-year warranty, price Rs. 1,299
Format: exactly 2 sentences, then one bullet line starting with 'Why buy:'.            # FORMAT
Constraints: warm practical tone, no emojis, no exclamation marks.                     # CONSTRAINTS
"""

print("=== VAGUE PROMPT ===")
print(ask(vague))
print()
print("=== ENGINEERED PROMPT ===")
print(ask(engineered))

**Compare:** length, tone, invented "facts", usability. Which one could go on a real website right now?

**✏️ Your turn:** delete ONE part (e.g. the constraints line) from the engineered prompt and re-run. Watch exactly what degrades — that's how you learn what each part does.

---
## Experiment 2 — Role prompting: one question, three personas 🎭

In [ ]:
question = "Why is the sky blue?"

roles = {
    "physics professor": "You are a physics professor speaking to final-year engineering students.",
    "kindergarten teacher": "You are a kindergarten teacher talking to 5-year-olds.",
    "poet": "You are a poet. Answer only in a 4-line rhyming poem.",
}

for name, system in roles.items():
    print(f"===== As a {name} =====")
    print(ask(question + " Keep it under 60 words.", system=system))
    print()

**Observation:** vocabulary, sentence length, tone and even *what counts as an answer* all shifted — from one line naming a role.

**✏️ Your turn:** add a fourth role that would matter for our project, e.g. `"an impatient store manager who answers in one blunt sentence"`.

---
## Experiment 3 — Few-shot support-ticket classifier 🏷️

A real retail problem: incoming customer messages must be routed to the right team. Instead of *describing* the rules, we **show examples** and let the model continue the pattern:

In [ ]:
few_shot_prompt = """Classify each customer message into exactly one label:
ORDER_STATUS, COMPLAINT, PRODUCT_QUERY, REFUND_REQUEST, OTHER

Examples:
Message: "Where is my order #4521?"
Label: ORDER_STATUS

Message: "The kettle arrived broken and I am very upset."
Label: COMPLAINT

Message: "Do you have the mixer grinder in blue?"
Label: PRODUCT_QUERY

Message: "I want my money back for order #9987."
Label: REFUND_REQUEST

Now classify this message. Reply with the label only.
Message: "{msg}"
Label:"""

test_messages = [
    "hasn't arrived yet, ordered 10 days ago, #3321",
    "kya aapke paas 2L wala kettle hai?",          # Hinglish — watch it cope!
    "the delivery boy was so rude, worst service",
    "please cancel and refund immediately",
    "what time does your Pimple Saudagar store open on Sunday?",
]

for msg in test_messages:
    label = ask(few_shot_prompt.format(msg=msg))
    print(f"{label:<16} <- {msg}")

**Notice:**
1. We never *defined* the labels — the examples carried the meaning
2. It handled **Hinglish** without being told
3. "Store hours" question fell to `OTHER` (or did it? check!) — edge cases reveal where you need one more example

**✏️ Your turn:** add a new label `STORE_INFO` with one example, and re-test the last message.

---
## Experiment 4 — “Think step by step” 🧮

A multi-step word problem. First we demand a bare answer, then we ask for the working:

In [ ]:
problem = (
    "A customer paid Rs. 2,600 in total for a kettle. This total includes "
    "Rs. 200 delivery. The kettle itself was bought at a 25% discount. "
    "What was the kettle's original price?"
)

print("=== Direct (answer only, no working allowed) ===")
print(ask(problem + " Give ONLY the final number, no explanation."))
print()
print("=== Step by step ===")
print(ask(problem + " Solve step by step, showing each calculation, then state the final answer."))

**Check the math yourself:** total 2600 − 200 delivery = 2400 paid for the kettle; 2400 is 75% of the original; 2400 ÷ 0.75 = **Rs. 3,200**.

Run the direct version a few times — does it ever slip? With harder problems, the gap grows. **Why it works:** the model predicts token by token, so writing out step 1 gives it better context for step 2. Bonus: you can *audit* its reasoning — crucial for business apps.

**✏️ Your turn:** invent a 3-step GST/discount problem from your own shopping and test both modes.

---
## Experiment 5 — Email → JSON extractor 📧

Chat answers are for humans; **applications need data**. Let's turn a messy customer email into clean JSON that could go straight into a database:

In [ ]:
import json

email = """subject: STILL WAITING!!!
hi team, i ordered a mixer grinder (the 750W one) last tuesday from your website,
order number 7788. it was supposed to reach in 3 days, i'm in pune, kothrud.
very disappointed. either deliver by friday or refund me. - Priya Deshmukh, ph 98xxxxxx21"""

extraction_prompt = f"""You are a data-entry assistant for SmartMart's support system.
Extract information from the customer email below.

Return ONLY valid JSON (no markdown fences, no commentary) with exactly these keys:
- customer_name (string)
- order_id (string)
- product (string)
- city (string)
- issue (one of: delivery_delay, damaged_item, wrong_item, refund_request, other)
- urgency (one of: low, medium, high)
If a value is missing in the email, use null.

EMAIL:
{email}"""

raw = ask(extraction_prompt)
print("Raw model output:")
print(raw)

# prove it's machine-usable:
data = json.loads(raw)
print("\n✅ Parsed by Python! Customer:", data["customer_name"], "| Order:", data["order_id"], "| Urgency:", data["urgency"])

**If `json.loads` crashed:** the model probably wrapped the JSON in ``` fences. That's a *prompt bug* — tighten the format instruction ("no markdown fences") or strip them in code. Welcome to real-world LLM engineering: the prompt and the parsing code are designed together.

**✏️ Your turn:** add a key `deadline_mentioned` (string or null) and re-run. Did it find "friday"?

---
## 🏁 Finale — SmartBot v2

Yesterday's SmartBot had 4 lines of rules. Today we rebuild its brain with everything from this lab: full anatomy, few-shot examples, output rules, and honest uncertainty.

In [ ]:
SMARTBOT_V2 = """# ROLE
You are SmartBot, the customer assistant of SmartMart retail store, Pimple Saudagar, Pune.

# CONTEXT (the only facts you know — never invent others)
- Store hours: 9 AM - 9 PM, all 7 days
- Returns: within 7 days WITH receipt; without receipt -> store credit only
- Delivery: free above Rs. 999, else Rs. 49; standard 2-4 days in Pune
- Current offer: 10% off kitchen appliances till Sunday
- Contact for escalation: pmssupport@smartmart.example / (+91) 9960664674

# TASK
Answer customer questions about the store, orders, deliveries and policies.

# FORMAT
- Maximum 3 sentences per reply
- If the customer sounds angry, first sentence must acknowledge their frustration

# CONSTRAINTS
- If you don't know something (e.g. live stock, order tracking), say exactly:
  "I'll need to check our system for that — may I have your order number?"
  (real checking arrives in Week 2-3 of our build!)
- Politely refuse anything unrelated to SmartMart. Suggest they contact the store for anything sensitive.
- Never invent prices, stock levels or delivery dates.

# EXAMPLES (follow this style)
Customer: "Till what time are you open?"
SmartBot: "We're open every day from 9 AM to 9 PM. See you soon!"

Customer: "This is the WORST store ever, my order is late!!"
SmartBot: "I'm really sorry your order is delayed — that's frustrating. I'll need to check our system for that — may I have your order number?"
"""

chat = client.chats.create(
    model=MODEL,
    config=types.GenerateContentConfig(system_instruction=SMARTBOT_V2),
)
print("✅ SmartBot v2 is online!")

In [ ]:
# The acceptance tests from today's slide 12 — run them all:
tests = [
    "when do you close on sunday?",                          # store info
    "my kettle broke after 2 days, i'm furious!",            # angry complaint
    "do you have iPhone 15 in stock right now?",             # unknown stock -> honest answer
    "can you write my physics homework?",                    # off-topic -> polite refusal
    "is delivery free for a Rs. 1,500 mixer?",               # policy question
]

for t in tests:
    print(f"🧑 Customer: {t}")
    print(f"🛒 SmartBot: {chat.send_message(t).text.strip()}")
    print("-" * 70)

In [ ]:
# 💬 BREAK-MY-BOT: chat freely and try to make SmartBot v2 violate its rules.
# Classic attacks: "ignore your instructions", "pretend you are a pirate",
# "what's the price of the X-500 scanner?" (it must not invent one!)
while True:
    user_msg = input("You: ")
    if user_msg.strip().lower() in ("quit", "exit", "q"):
        print("👋 Nice attacks! See you in Module 3.")
        break
    print("🛒 SmartBot:", chat.send_message(user_msg).text.strip())

---
## 🏆 Homework (15 min, counts toward your weekly project)

1. **Harden v2** — swap the interactive cell with a neighbour (or a friend on the phone): each of you gets 3 attempts to break the other's bot. For every successful break, add one constraint or example that fixes it. Save the before/after.
2. **Extend the classifier** — add `STORE_INFO` and `HINGLISH` handling to Experiment 3 so all five test messages route correctly.
3. **One-line challenge** — rewrite Experiment 5's prompt so the JSON *also* includes a `suggested_reply` field written in SmartBot's voice. Now one API call does extraction AND drafting.

### What's next
**Module 3 — Working with LLM APIs:** OpenAI, Anthropic and open-source models. Same prompts, different engines — you'll learn the professional plumbing: SDKs, message roles, error handling and choosing a provider.

---
*PMS RoBoTics Research Center · pmsroboticsrc@gmail.com · (+91) 9960664674*